# MLflow Example: Tracking Two Model Training Runs

In [1]:
! pip3 install sklearn

Defaulting to user installation because normal site-packages is not writeable
  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
      
      More information is available at
      https://github.com/scikit-learn/sklearn-

In [2]:
# === Model Training and Experimentation ===

# Import necessary libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits


In [3]:

# Load dataset
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

# Define a function to train a model and return its accuracy
def train_model(n_estimators, max_depth):
    # Train a RandomForestClassifier
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)

    # Make predictions
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    print(f"Model with n_estimators={n_estimators}, max_depth={max_depth} achieved accuracy={accuracy:.4f}")
    return model, accuracy

In [4]:

# Train models with different hyperparameter configurations
print("Training Model 1...")
model_1, acc_1 = train_model(n_estimators=20, max_depth=5)

Training Model 1...
Model with n_estimators=20, max_depth=5 achieved accuracy=0.9389


In [5]:
print("\nTraining Model 2...")
model_2, acc_2 = train_model(n_estimators=100, max_depth=10)



Training Model 2...
Model with n_estimators=100, max_depth=10 achieved accuracy=0.9722


Now adding MLFlow

In [6]:
! pip3 install mlflow

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: C:\Users\arman\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits

In [8]:
!pip3 install dagshub

Defaulting to user installation because normal site-packages is not writeable
  Using cached graphql_core-3.3.2-py3-none-any.whl.metadata (11 kB)
Using cached graphql_core-3.3.2-py3-none-any.whl (366 kB)
  Attempting uninstall: graphql-core
    Found existing installation: graphql-core 3.2.14
    Uninstalling graphql-core-3.2.14:
      Successfully uninstalled graphql-core-3.2.14


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
graphene 3.4.3 requires graphql-core<3.3,>=3.1, but you have graphql-core 3.3.2 which is incompatible.
graphql-relay 3.2.0 requires graphql-core<3.3,>=3.2, but you have graphql-core 3.3.2 which is incompatible.

[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: C:\Users\arman\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [9]:
import dagshub
dagshub.init(repo_owner='ArmandColonnaW', repo_name='my-dvc-project', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Accessing as ArmandColonnaW

Initialized MLflow to track repo "ArmandColonnaW/my-dvc-project"

Repository ArmandColonnaW/my-dvc-project initialized!

🏃 View run bustling-sloth-221 at: https://dagshub.com/ArmandColonnaW/my-dvc-project.mlflow/#/experiments/0/runs/8c221d684220448fb6d3a6302909ae77
🧪 View experiment at: https://dagshub.com/ArmandColonnaW/my-dvc-project.mlflow/#/experiments/0


In [10]:
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

In [11]:
# Set MLflow experiment name. This will get created if it doesn't exist
experiment_name = "RandomForest_V1.0.1"
mlflow.set_experiment(experiment_name)


<Experiment: artifact_location='mlflow-artifacts:/c80dc457256d4dfdb01ce636964813fc', creation_time=1794845294554, experiment_id='0', last_update_time=1794845294554, lifecycle_stage='active', name='RandomForest_V1.0.1', tags={}>

In [12]:
# Define a function to train a model, log parameters and metrics to MLflow
def train_and_log_model(n_estimators, max_depth, run_name):
    with mlflow.start_run(run_name=run_name):
        # Train a RandomForestClassifier
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Make predictions
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)

        # Log parameters, metrics, and the model itself
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_metric("accuracy", accuracy)
        mlflow.sklearn.log_model(sk_model=model, artifact_path="model")

        print(f"Logged RandomForest model with n_estimators={n_estimators}, max_depth={max_depth}, accuracy={accuracy:.4f}")


In [13]:
# Train and log Model 1
print("Training and Logging Model 1...")
train_and_log_model(
    n_estimators=20, max_depth=5, run_name="random_forest_model_run_1"
)

Training and Logging Model 1...


2026/11/18 16:12:35 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Logged RandomForest model with n_estimators=20, max_depth=5, accuracy=0.9389
🏃 View run random_forest_model_run_1 at: https://dagshub.com/ArmandColonnaW/my-dvc-project.mlflow/#/experiments/0/runs/89ca60048e2b48048bbe40f56cf88bd6
🧪 View experiment at: https://dagshub.com/ArmandColonnaW/my-dvc-project.mlflow/#/experiments/0


In [14]:
# Train and log Model 2
print("\nTraining and Logging Model 2...")
train_and_log_model(
    n_estimators=100, max_depth=10, run_name="random_forest_model_run_V1.0.2"
)


Training and Logging Model 2...


2026/11/18 16:12:54 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Logged RandomForest model with n_estimators=100, max_depth=10, accuracy=0.9722
🏃 View run random_forest_model_run_V1.0.2 at: https://dagshub.com/ArmandColonnaW/my-dvc-project.mlflow/#/experiments/0/runs/31abdbebfe954e8aabe2725ed45d148b
🧪 View experiment at: https://dagshub.com/ArmandColonnaW/my-dvc-project.mlflow/#/experiments/0


In [15]:
# Instructions to visualize results
print("\nTo view the results, run the following command in your terminal:")
print("mlflow ui")
print("Then navigate to http://127.0.0.1:5000 to explore the experiment results.")


To view the results, run the following command in your terminal:
mlflow ui
Then navigate to http://127.0.0.1:5000 to explore the experiment results.
